# Orion extractor: NuExtract 4B vs 8B on a Kaggle GPU

Runs the extractor (GLiNER2 + BookNLP + NuExtract through Ollama) with each model in `MODELS` and saves:

1. **Gold evaluation**: the 4 annotated stories in `extractor/data/gold`, scored (mentions, coreference, facts,
   relationships, events), with a side-by-side table at the end.
2. **End-to-end stories**: every chapter of `docs/e2e_story_gull_point.txt` and `docs/e2e_story_river_sall.txt`,
   split exactly as the backend splits them. The raw extractor JSONs are scored locally through the backend
   (contradictions, relationships, facts), so no database is needed here.

**Before running:** Settings → Accelerator **GPU T4 x2** (not P100: recent PyTorch dropped it), and
Settings → **Internet on** (needs a phone-verified account). Then *Run All*. About 15 min of setup (downloads) and
~10 min per model.

**After running:** download `orion_results.zip` from the Output panel (right side, `/kaggle/working`).

In [ ]:
REPO = "https://github.com/SkySwordSaint26/Orion---World-State-Engine.git"
BRANCH = "feat/kaggle-model-comparison"     # use "main" once this branch is merged
MODELS = {                                  # Ollama name -> (Hugging Face repo, GGUF file)
    "nuextract2-4b": ("numind/NuExtract-2.0-4B-GGUF", "NuExtract-2.0-4B-Q4_K_M.gguf"),      # what runs locally
    "nuextract2-8b": ("numind/NuExtract-2.0-8B-GGUF", "NuExtract-2.0-8B-Q4_K_M.gguf"),      # same compression
    "nuextract2-8b-q8": ("numind/NuExtract-2.0-8B-GGUF", "NuExtract-2.0-8B-Q8_0.gguf"),     # near full quality
}
STORIES = ["docs/e2e_story_gull_point.txt", "docs/e2e_story_river_sall.txt"]
WORK = "/kaggle/working"

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir(WORK)
!rm -rf orion && git clone -q --depth 1 -b {BRANCH} {REPO} orion && git -C orion log --oneline -1
EXT = f"{WORK}/orion/extractor"

## 2. Python 3.12 environment

A separate venv with the extractor's pinned versions, so Kaggle's preinstalled packages can't clash with them.

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 {WORK}/venv
PY = f"{WORK}/venv/bin/python"
!uv pip install -q -p {PY} -r {EXT}/requirements.txt
!uv pip install -q -p {PY} --no-deps booknlp==1.0.8
!uv pip install -q -p {PY} https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
# The pinned torch must see the GPU; if Kaggle's driver is too old for its CUDA build, fall back to a CUDA 12.6 build.
check = lambda: subprocess.run([PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                               capture_output=True, text=True).stdout.strip()
print(check())
if not check().endswith("True"):
    !uv pip install -q -p {PY} --reinstall torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
    print(check())
assert check().endswith("True"), "torch cannot see the GPU: is the accelerator set to GPU T4 x2?"

## 3. Ollama and the NuExtract models

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())

In [ ]:
# Same Modelfile as locally (extractor/nuextract.Modelfile: ChatML template, temperature 0), pointed at each GGUF.
template = open(f"{EXT}/nuextract.Modelfile").read().split("\n", 1)[1]
for name, (repo, file) in MODELS.items():
    path = subprocess.run([PY, "-c", f"from huggingface_hub import hf_hub_download as d; print(d({repo!r}, {file!r}))"],
                          capture_output=True, text=True, check=True).stdout.strip()
    open(f"{WORK}/{name}.Modelfile", "w").write(f"FROM {path}\n{template}")
    subprocess.run(["ollama", "create", name, "-f", f"{WORK}/{name}.Modelfile"], check=True, capture_output=True)
    print("registered", name, "from", file)
!ollama list

## 4. Smoke test

One chapter with the 4B model. The first run also downloads GLiNER2 and BookNLP (~3 GB).

In [ ]:
import runpy, pathlib, json, shutil
split_into_chapters = runpy.run_path(f"{WORK}/orion/backend/app/utils/text_splitter.py")["split_into_chapters"]
OUT = pathlib.Path(f"{WORK}/results"); OUT.mkdir(exist_ok=True)

def run(args, model, **kw):
    t = time.time()
    p = subprocess.run([PY, "-m", "extractor", *args], cwd=EXT, env={**os.environ, "ORION_LLM_MODEL": model},
                       capture_output=True, text=True, **kw)
    if p.returncode:
        print(p.stdout[-3000:], p.stderr[-3000:]); p.check_returncode()
    return p, time.time() - t

chapter1 = split_into_chapters(open(f"{WORK}/orion/{STORIES[0]}").read())[0][2]
pathlib.Path(f"{WORK}/smoke.txt").write_text(chapter1)
_, seconds = run(["run", f"{WORK}/smoke.txt", f"{WORK}/smoke.json"], "nuextract2-4b")
doc = json.load(open(f"{WORK}/smoke.json"))
print(f"{seconds:.0f}s:", {k: len(v) for k, v in doc.items() if isinstance(v, list)})

## 5. Gold evaluation and end-to-end stories, per model

In [ ]:
timings = {}
for model in MODELS:
    print(f"=== {model}")
    gold_out = OUT / model / "gold"
    p, seconds = run(["predict", "--gold-dir", "data/gold", "--out-dir", str(gold_out)], model)
    print(f"gold: {seconds:.0f}s")
    run(["score", "data/gold", str(gold_out), "--json", str(OUT / model / "gold_scores.json")], model)
    for story in STORIES:
        name = pathlib.Path(story).stem
        for number, title, text in split_into_chapters(open(f"{WORK}/orion/{story}").read()):
            folder = OUT / model / name; folder.mkdir(parents=True, exist_ok=True)
            (folder / f"ch{number}.txt").write_text(text)
            _, seconds = run(["run", str(folder / f"ch{number}.txt"), str(folder / f"ch{number}.json")], model)
            timings[f"{model}/{name}/ch{number}"] = round(seconds, 1)
            print(f"{name} ch{number}: {seconds:.0f}s")
json.dump(timings, open(OUT / "timings.json", "w"), indent=1)

## 6. Gold scores side by side (micro F1)

In [ ]:
scores = {m: json.load(open(OUT / m / "gold_scores.json"))["micro"] for m in MODELS}
metrics = [k for k, v in scores[next(iter(MODELS))].items() if isinstance(v, dict) and "f1" in v]
print(f"{'metric':26}" + "".join(f"{m:>18}" for m in MODELS))
for k in metrics:
    print(f"{k:26}" + "".join(f"{scores[m][k]['f1']:>18.3f}" for m in MODELS))

In [ ]:
shutil.make_archive(f"{WORK}/orion_results", "zip", OUT)
print("download from the Output panel:", f"{WORK}/orion_results.zip")